# Notebook 07 — Diagnosing Slow Queries: Your Troubleshooting Playbook
## Before You Escalate — Run These Diagnostics

**What you'll learn**: A step-by-step approach to figure out WHY your query is slow and WHAT to ask for.

| Symptom | Likely Cause | Your Action |
|---|---|---|
| Date-range report is slow | Data not loaded in date order | Ask: *"Is this table loaded chronologically?"* |
| Customer lookup scans everything | No clustering on account_id | Ask: *"Can we cluster by account_id?"* |
| Point lookup reads full table | No search optimization | Ask: *"Can we enable SOS on transaction_id?"* |
| Complex report spills to disk | Warehouse too small | Ask: *"Can I use MEDIUM for monthly reports?"* |
| Ad-hoc scan is slow | No query acceleration | Ask: *"Can we enable QAS on our analytics warehouse?"* |

In [ ]:
USE ROLE SYSADMIN;
USE WAREHOUSE HOL_XS;
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

---
## Step 1: Find Your Slowest Queries (Last 2 Hours)

This query shows your recent slow queries — no special admin access needed.

In [ ]:
-- Your slowest queries from this session (no admin access needed)
SELECT
    query_id,
    SUBSTR(query_text, 1, 80) AS query_preview,
    warehouse_name,
    total_elapsed_time / 1000 AS elapsed_sec,
    bytes_scanned / (1024*1024*1024) AS gb_scanned
FROM TABLE(SNOWFLAKE.INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(hour, -2, CURRENT_TIMESTAMP()),
    END_TIME_RANGE_END => CURRENT_TIMESTAMP(),
    RESULT_LIMIT => 50
))
WHERE query_type = 'SELECT'
  AND total_elapsed_time > 1000
ORDER BY total_elapsed_time DESC
LIMIT 10;

---
## Step 2: Find Queries That Scan Too Much Data

If a query reads gigabytes but returns only a few rows, it's a candidate for clustering or SOS.

In [ ]:
-- Queries scanning lots of data relative to results
SELECT
    query_id,
    SUBSTR(query_text, 1, 80) AS query_preview,
    bytes_scanned / (1024*1024*1024) AS gb_scanned,
    rows_produced,
    total_elapsed_time / 1000 AS elapsed_sec
FROM TABLE(SNOWFLAKE.INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(hour, -2, CURRENT_TIMESTAMP()),
    END_TIME_RANGE_END => CURRENT_TIMESTAMP(),
    RESULT_LIMIT => 50
))
WHERE query_type = 'SELECT'
  AND bytes_scanned > 1073741824
  AND rows_produced < 1000
ORDER BY bytes_scanned DESC
LIMIT 10;

---
## Step 3: Find Queries That Spill (Need Bigger Warehouse)

In [ ]:
-- Queries that ran out of memory (spilled to disk)
SELECT
    query_id,
    SUBSTR(query_text, 1, 80) AS query_preview,
    warehouse_size,
    bytes_spilled_to_local_storage / (1024*1024*1024) AS spill_local_gb,
    bytes_spilled_to_remote_storage / (1024*1024*1024) AS spill_remote_gb,
    total_elapsed_time / 1000 AS elapsed_sec
FROM TABLE(SNOWFLAKE.INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(hour, -2, CURRENT_TIMESTAMP()),
    END_TIME_RANGE_END => CURRENT_TIMESTAMP(),
    RESULT_LIMIT => 50
))
WHERE (bytes_spilled_to_local_storage > 0 OR bytes_spilled_to_remote_storage > 0)
ORDER BY bytes_spilled_to_remote_storage DESC
LIMIT 10;

---
## Your Decision Tree: "My Query Is Slow — What Do I Do?"

```
My query is slow. What should I check?
│
├── Is it scanning way more data than it returns?
│   ├── Am I filtering by a date range? → Check: is the table loaded chronologically? (Notebook 02)
│   ├── Am I looking up 1 record by ID? → Request: Search Optimization (Notebook 05)
│   └── Am I filtering by customer/account? → Request: Cluster key on that column (Notebook 03)
│
├── Is it spilling to disk? (Query Profile shows spilling)
│   └── Request: bigger warehouse for this workload (Notebook 04)
│
├── Is it a large scan with a selective filter?
│   └── Request: Query Acceleration Service (Notebook 04)
│
└── None of the above?
    └── Check YOUR query: SELECT *, late filters, cache-defeating functions (Notebook 06)
```

---
## Step 4: Session Cost Summary

In [ ]:
-- Credits consumed during this HOL session
SELECT
    warehouse_name,
    COUNT(*) AS queries_run,
    SUM(total_elapsed_time) / 1000 AS total_runtime_sec,
    SUM(bytes_scanned) / (1024*1024*1024) AS total_gb_scanned
FROM TABLE(SNOWFLAKE.INFORMATION_SCHEMA.QUERY_HISTORY(
    END_TIME_RANGE_START => DATEADD(hour, -3, CURRENT_TIMESTAMP()),
    END_TIME_RANGE_END => CURRENT_TIMESTAMP(),
    RESULT_LIMIT => 200
))
WHERE query_type = 'SELECT'
GROUP BY 1
ORDER BY total_gb_scanned DESC;

---
## Summary: Your Performance Engineering Cheat Sheet

**Things you control (do these first — they're free):**
1. Never use `SELECT *` — only pull the columns you need
2. Filter early — reduce rows before joining
3. Use fixed dates in dashboards to leverage result caching
4. Check the Query Profile after every new report query

**Things to request from your platform team:**
1. **Clustering** — when customer/account lookups scan the full table
2. **Search Optimization** — when point lookups or LIKE searches are slow
3. **Bigger warehouse** — when your query spills to disk
4. **Query Acceleration** — when ad-hoc scans are slow

**How to make a strong request:**
- Share the **Query ID** and **Query Profile screenshot**
- Show the **bytes scanned** vs **rows returned** gap
- Reference which feature would help (clustering, SOS, sizing, QAS)

---
## Cleanup (optional)

In [ ]:
-- Uncomment to tear down the entire HOL environment
-- DROP DATABASE IF EXISTS JPMC_PERF_HOL;
-- DROP WAREHOUSE IF EXISTS HOL_XS;
-- DROP WAREHOUSE IF EXISTS HOL_M;
-- DROP WAREHOUSE IF EXISTS HOL_L;
-- DROP WAREHOUSE IF EXISTS HOL_XL;
-- DROP WAREHOUSE IF EXISTS HOL_NO_QAS;
-- DROP WAREHOUSE IF EXISTS HOL_QAS;